# ڕاهێنانی AI وەرگێڕی بادینی

ئەم نۆتبووکە مۆدێلی بادینی arkanit/badini-nllb-translator ڕاهێنانی زیاتر پێ دەکات؛ مۆدێلی نوێ لە سفرەوە دروست ناکات. لە Google Colab بیکەرەوە، لە Runtime → Change runtime type، **GPU** هەڵبژێرە و خانەکان بە ڕیز جێبەجێ بکە. GPUـی بەخۆڕایی دڵنیانەکراوە؛ لەوانەیە بەردەست نەبێت یان خەرجی هەبێت.

مۆدێلی بنەڕەت CC-BY-NC-4.0 ـە؛ داتای ناوخۆی فەرهەنگ MIT ـە. ئەم نۆتبووکە بەبێ داتای زیادە تەنها ئینگلیزی و عەرەبی → بادینی ڕاهێنان دەکات. بۆ سۆرانی پێویستە داتای هاوتای پشکنراو زیاد بکرێت. هیچ دڵنیاییەک لە کوالێتی وەرگێڕان نییە.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("تکایە لە Runtime → Change runtime type GPU هەڵبژێرە و دووبارە ئەم خانەیە جێبەجێ بکە.")
print("GPU:", torch.cuda.get_device_name(0))


## ١. پەیوەندی بە Google Drive
مۆدێل و خاڵەکانی بەردەوامبوونەوە لە Drive پاشەکەوت دەکرێن. چەند گیگابایت بۆ مۆدێل و checkpoint پێویستە.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## ٢. دامەزراندنی پێداویستییەکان


In [ ]:
%pip install -q "transformers==4.46.3" "accelerate>=0.26,<2" "sentencepiece>=0.2" "protobuf>=3.20"


## ٣. ئامادەکردنی داتا و کۆد
فەرهەنگەکە لە GitHub دادەگیرێت، و مۆدێلی بنەڕەت لە Hugging Face دادەگیرێت.


In [ ]:
import io, tarfile, urllib.request
from pathlib import Path

work = Path('/content/badini-training')
data_dir = work / 'training_data'
data_dir.mkdir(parents=True, exist_ok=True)
url = 'https://codeload.github.com/Jad-deeb/Kurdish-behdini-language-resources/tar.gz/d03e6f7fe0e92c92e3cbf240928c5b0ba7674851'
filenames = [
    'Dictionary_English-BehdiniLatin-BehdiniArabic-Arabic-Transliteration.csv',
    'Expressions_BehdiniArabic-BehdiniLatin-English.csv',
    'Proverbs_BehdiniArabic-BehdiniLatini-English.csv',
    'Verbs_Verb-Tense-Person-English-BehdiniLatin-BehdiniArabic.csv',
    'LICENSE',
]
with urllib.request.urlopen(url, timeout=120) as response:
    archive = response.read()
with tarfile.open(fileobj=io.BytesIO(archive), mode='r:gz') as tar:
    for filename in filenames:
        matches = [m for m in tar.getmembers() if m.isfile() and m.name.split('/')[-1] == filename]
        if len(matches) != 1:
            raise RuntimeError(f'Missing or ambiguous data file: {filename}')
        (data_dir / filename).write_bytes(tar.extractfile(matches[0]).read())
print('داتا ئامادەیە:', data_dir)


In [ ]:
# The training program is included in this notebook; no GitHub push is required.
training_script = '"""Fine-tune a local, personal NLLB translation model on parallel Badini data."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\n\nBASE_MODEL = "arkanit/badini-nllb-translator"\nLANGUAGES = {"en": "eng_Latn", "ar": "arb_Arab", "sorani": "ckb_Arab", "badini": "ckb_Arab"}\nDATA = Path(__file__).resolve().parent / "training_data"\n\n\ndef read_pairs(extra=None):\n    pairs = set()\n    def add(language, source, target):\n        source, target = (source or "").strip(), (target or "").strip()\n        if language in ("en", "ar", "sorani") and source and target:\n            pairs.add((language, source, target))\n    files = [\n        ("Dictionary_English-BehdiniLatin-BehdiniArabic-Arabic-Transliteration.csv", "Kurdish (Arabic Script)", True),\n        ("Expressions_BehdiniArabic-BehdiniLatin-English.csv", "Kurdish (Arabic Script)", False),\n        ("Proverbs_BehdiniArabic-BehdiniLatini-English.csv", "Kurdish Arabic", False),\n        ("Verbs_Verb-Tense-Person-English-BehdiniLatin-BehdiniArabic.csv", "Kurdish (Arabic)", False),\n    ]\n    for filename, target, has_arabic in files:\n        with (DATA / filename).open(encoding="utf-8-sig", newline="") as f:\n            for row in csv.DictReader(f):\n                add("en", row.get("English"), row.get(target))\n                if has_arabic:\n                    add("ar", row.get("Arabic"), row.get(target))\n    # Published teacher outputs are examples, not live model inference.\n    teacher_examples = DATA / "badini_model_card_examples.csv"\n    with teacher_examples.open(encoding="utf-8-sig", newline="") as f:\n        for row in csv.DictReader(f):\n            add(row.get("source_lang"), row.get("source_text"), row.get("badini"))\n    if extra:\n        with Path(extra).open(encoding="utf-8-sig", newline="") as f:\n            reader = csv.DictReader(f)\n            if not reader.fieldnames or not {"source_lang", "source_text", "badini"}.issubset(reader.fieldnames):\n                raise ValueError("Extra CSV must have source_lang,source_text,badini columns.")\n            for line, row in enumerate(reader, 2):\n                lang = (row.get("source_lang") or "").strip()\n                if lang not in ("en", "ar", "sorani") or not (row.get("source_text") or "").strip() or not (row.get("badini") or "").strip():\n                    raise ValueError(f"Invalid training row {line}.")\n                add(lang, row["source_text"], row["badini"])\n    return sorted(pairs)\n\n\ndef split_pairs(pairs, seed=42):\n    # Keep all translations of the same source on the same side of the split.\n    groups = {}\n    for lang, source, target in pairs:\n        groups.setdefault((lang, source.casefold()), []).append((lang, source, target))\n    def is_heldout(key):\n        lang, source = key\n        digest = hashlib.sha256(f"{seed}\\0{lang}\\0{source}".encode("utf-8")).digest()\n        return int.from_bytes(digest[:8], "big") < (1 << 64) // 20\n    train = [row for key in sorted(groups) if not is_heldout(key) for row in groups[key]]\n    test = [row for key in sorted(groups) if is_heldout(key) for row in groups[key]]\n    return train, test\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--data", help="Optional UTF-8 CSV: source_lang,source_text,badini")\n    parser.add_argument("--base", default=BASE_MODEL, help="Starting model ID or previously trained personal model directory")\n    parser.add_argument("--output", default="models/my-badini-ai")\n    parser.add_argument("--epochs", type=float, default=3)\n    parser.add_argument("--batch-size", type=int, default=2)\n    parser.add_argument("--max-length", type=int, default=128)\n    parser.add_argument("--resume", help="Resume from a Trainer checkpoint directory")\n    args = parser.parse_args()\n    if args.epochs <= 0 or args.batch_size < 1 or args.max_length < 2:\n        parser.error("epochs, batch size and maximum length must be positive")\n\n    import torch\n    from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,\n                              Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed)\n    set_seed(42)\n    train, heldout = split_pairs(read_pairs(args.data))\n    if not train:\n        raise ValueError("No training pairs found.")\n    print(f"Training: {len(train)} pairs; held-out: {len(heldout)} pairs.", flush=True)\n    tokenizer = AutoTokenizer.from_pretrained(args.base)\n    model = AutoModelForSeq2SeqLM.from_pretrained(args.base)\n\n    class Pairs(torch.utils.data.Dataset):\n        def __init__(self, rows):\n            self.rows = rows\n        def __len__(self):\n            return len(self.rows)\n        def __getitem__(self, index):\n            lang, source, target = self.rows[index]\n            tokenizer.src_lang = LANGUAGES[lang]\n            tokenizer.tgt_lang = LANGUAGES["badini"]\n            inputs = tokenizer(source, truncation=True, max_length=args.max_length)\n            inputs["labels"] = tokenizer(text_target=target, truncation=True, max_length=args.max_length)["input_ids"]\n            return inputs\n\n    output = Path(args.output)\n    output.mkdir(parents=True, exist_ok=True)\n    with (output / "heldout.csv").open("w", encoding="utf-8", newline="") as f:\n        writer = csv.writer(f)\n        writer.writerow(["source_lang", "source_text", "badini"])\n        writer.writerows(heldout)\n    options = Seq2SeqTrainingArguments(\n        output_dir=str(output / "checkpoints"), num_train_epochs=args.epochs,\n        per_device_train_batch_size=args.batch_size, gradient_accumulation_steps=4,\n        learning_rate=2e-5, save_strategy="steps", save_steps=250, save_total_limit=1,\n        logging_steps=50, report_to="none", dataloader_num_workers=0,\n        fp16=torch.cuda.is_available(),\n    )\n    trainer = Seq2SeqTrainer(model=model, args=options, train_dataset=Pairs(train),\n                            data_collator=DataCollatorForSeq2Seq(tokenizer, model=model))\n    trainer.train(resume_from_checkpoint=args.resume)\n    trainer.save_model(str(output))\n    tokenizer.save_pretrained(str(output))\n    previous_metadata = Path(args.base) / "badini_model.json"\n    previous_languages = set()\n    if previous_metadata.is_file():\n        previous_languages = set(json.loads(previous_metadata.read_text(encoding="utf-8"))["source_languages"])\n    (output / "badini_model.json").write_text(json.dumps({"source_languages": sorted(previous_languages | {r[0] for r in train}), "base_model": args.base}, indent=2), encoding="utf-8")\n    print(f"Your trained model is saved in {output}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
(work / "train.py").write_text(training_script, encoding="utf-8")
(data_dir / "badini_model_card_examples.csv").write_text('source_lang,source_text,badini\nen,How are you? Are you doing well?,تو چەوانی باشی\nen,Where do you live?,تو ل کیڤە دژی\n', encoding="utf-8")


## ٤. هەڵبژاردنەکان
ئەگەر داتای تایبەتی هەیە، CSV لە Drive دابنێ بە ستوونەکانی `source_lang,source_text,badini` و ڕێڕەوی لێرە بنووسە. بۆ سۆرانی `source_lang` دەبێت `sorani` بێت. بەبێ ئەوە سۆرانی ڕاهێنان ناکرێت. نموونەکان پێش ڕاهێنان بپشکنە.

بۆ بەردەوامبوونەوە دوای پچڕانی Colab، `RESUME` بکە بە ڕێڕەوی checkpoint؛ تەنها ئەگەر داتا و هەڵبژاردنەکان وەک ڕاهێنانی پێشوو بێن.


In [ ]:
OUTPUT = '/content/drive/MyDrive/badini-ai/model'
EXTRA_CSV = ''  # e.g. '/content/drive/MyDrive/badini-ai/my-pairs.csv'
BASE = ''       # Optional previous personal model directory on Drive
RESUME = ''     # e.g. '/content/drive/MyDrive/badini-ai/model/checkpoints/checkpoint-250'
EPOCHS = 3
BATCH_SIZE = 2


## ٥. دەستپێکردنی ڕاهێنان
ماوەی کارکردن بە GPU و سنووری Colab دەگۆڕێت. ئەگەر هەڵەی بیرگە ڕوویدا، `BATCH_SIZE` بکە بە ١ و دووبارە هەوڵ بدە.


In [ ]:
import subprocess, sys
command = [sys.executable, str(work / 'train.py'), '--output', OUTPUT, '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE)]
if BASE:
    command += ['--base', BASE]
if EXTRA_CSV:
    command += ['--data', EXTRA_CSV]
if RESUME:
    command += ['--resume', RESUME]
subprocess.run(command, cwd=str(work), check=True)
print('مۆدێل پاشەکەوت کرا لە:', OUTPUT)


## ٦. تاقیکردنەوەی مۆدێلی پاشەکەوتکراو
زمانی دەقەکە هەڵبژێرە. سۆرانی تەنها کاتێک کار دەکات کە بە داتای سۆرانی ڕاهێنرابێت. ئەمە تاقیکردنەوەی کوالێتی نییە؛ ئەنجامەکان بە خەڵکی بادینیزان بپشکنە.


In [ ]:
import json, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

source_language = 'en'  # 'en', 'ar', or 'sorani'
text = 'Hello, how are you?'
language_codes = {'en': 'eng_Latn', 'ar': 'arb_Arab', 'sorani': 'ckb_Arab'}
metadata = json.loads((Path(OUTPUT) / 'badini_model.json').read_text(encoding='utf-8'))
if source_language not in metadata['source_languages']:
    raise ValueError('ئەم زمانە لە داتای ڕاهێناندا نییە.')
tokenizer = AutoTokenizer.from_pretrained(OUTPUT)
model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT).to('cuda').eval()
tokenizer.src_lang = language_codes[source_language]
inputs = tokenizer(text, return_tensors='pt').to('cuda')
with torch.inference_mode():
    output = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids('ckb_Arab'), max_new_tokens=128)
print(tokenizer.decode(output[0], skip_special_tokens=True))
